# Stage 6. Extract features

## Notebook Information

**Purpose:** Extract per-object morphology, intensity, and texture features from segmented field-of-view images.

**Workflow summary:** Load configuration and metadata, open illumination-corrected FOV images and matching segmentation masks, remove edge-touching labels, compute regionprops/custom intensity features plus Hessian and structure-tensor features, then save one feature table per FOV and update processing metadata.

**Run scope:** Processes all rows in the selected metadata dataframe by default. The expected input is the Part 5 metadata containing illumination-correction and segmentation file references.

**Reproducibility:** Use the configuration/parameter cell as the source of truth for paths and run parameters. For a fresh execution, run sections in order and make sure the metadata input points to the intended Part 5 metadata file rather than a previously generated Part 6 metadata file.

### Authors

| Name | Affiliation |
|---|---|
| Alessandro Ulivi | Infectious Diseases Imaging Platform, Center for Integrative Infectious Disease Research, Heidelberg |
| Edwin Carreno | Scientific Software Center, Heidelberg |
| Christine Schultz | Scientific Software Center, Heidelberg |
| Name Surname | Infectious Diseases Imaging Platform, Center for Integrative Infectious Disease Research, Heidelberg |

## 1. Setup

### 1.1 Imports

In [ ]:
# Built-in imports
import logging

# Third-party imports
import pandas as pd

# Package imports
from acid.utils.filesystem.filesystem import create_output_directories
from acid.utils.metadata.loading import load_metadata


### 1.2 Configure logging

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

### 1.3 Load configuration

In [ ]:
from acid.config import load_config, prepare_output

# Use the same file in every stage. Set to None for defaults only.
CONFIG_FILE = None  # "configs/exp_001.yaml"
config = load_config(CONFIG_FILE)
# Save effective settings, or reject changes to an existing output folder.
prepare_output(config)

In [ ]:
paths_cfg = config.shared.paths

# Metadata config
metadata_cfg = config.feature_extraction.metadata

## 2. Input Data Preparation

### 2.1 Create output directories

In [ ]:
output_path, _ = create_output_directories(
    output_directory=paths_cfg.feature_tables_dir,
    enable_secondary_output=False,
)

### 2.2 Load metadata dataframe

In [ ]:
metadata_df, metadata_file_name = load_metadata(
    metadata_config=config["feature_extraction"]["metadata"]
)

## 3. Extract features

- Preprocess image and segmentation mask
- Extract features
- Save results

In [ ]:
from acid.feature_extraction.pipeline import (
    extract_features_batch,
    update_metadata_with_feature_results,
)
from acid.utils.metadata.saving import save_metadata_dataframe

In [ ]:
feature_extraction_cfg = config.feature_extraction

results = extract_features_batch(
    metadata_df=metadata_df, config=feature_extraction_cfg, paths=paths_cfg
)

In [ ]:
results_df = pd.DataFrame.from_records(results)
results_df

## 4. Save results

In [ ]:
metadata_df_updated = update_metadata_with_feature_results(
    metadata_df=metadata_df,
    results=results,
    dataframe_columns=config.feature_extraction.metadata.dataframe_columns,
)

metadata_df_updated.info()

In [ ]:
metadata_path = save_metadata_dataframe(
    metadata_df=metadata_df_updated,
    metadata_config=metadata_cfg,
    project_name=config.project_identity.project_name,
)

logging.info(f"Metadata path: {metadata_path}")
logging.info("processing metadata saved")
logging.info("finished")

# End Notebook